In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 02. Data Preprocessing Pipeline\n",
    "## NLP Chatbot Project - Banking77 Dataset\n",
    "\n",
    "This notebook covers:\n",
    "- Text cleaning and normalization\n",
    "- Tokenization\n",
    "- Stopword removal\n",
    "- Lemmatization\n",
    "- Feature engineering\n",
    "- TF-IDF vectorization\n",
    "- Train/Val/Test split preparation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import sys\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Add src to path\n",
    "sys.path.append('../src')\n",
    "\n",
    "# Import custom modules\n",
    "from preprocessor import TextPreprocessor, TextVectorizer, prepare_data\n",
    "\n",
    "# Set style\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (12, 6)\n",
    "\n",
    "print(\"✅ Imports successful!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load Raw Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load raw data\n",
    "df = pd.read_csv('../data/raw_data.csv')\n",
    "\n",
    "print(f\"Dataset shape: {df.shape}\")\n",
    "print(f\"\\nColumns: {df.columns.tolist()}\")\n",
    "print(f\"\\nFirst few rows:\")\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Initialize Preprocessor"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize text preprocessor\n",
    "preprocessor = TextPreprocessor(\n",
    "    remove_stopwords=True,\n",
    "    lemmatize=True\n",
    ")\n",
    "\n",
    "print(\"✅ Preprocessor initialized\")\n",
    "print(f\"   - Stopword removal: {preprocessor.remove_stopwords}\")\n",
    "print(f\"   - Lemmatization: {preprocessor.lemmatize}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Visualize Preprocessing Effects"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Take sample texts\n",
    "sample_texts = df['text'].head(10).tolist()\n",
    "\n",
    "# Preprocess\n",
    "processed_texts = [preprocessor.preprocess(text) for text in sample_texts]\n",
    "\n",
    "# Display comparison\n",
    "print(\"=\"*80)\n",
    "print(\"PREPROCESSING EXAMPLES\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "for i, (original, processed) in enumerate(zip(sample_texts, processed_texts), 1):\n",
    "    print(f\"\\n{i}. ORIGINAL:\")\n",
    "    print(f\"   {original}\")\n",
    "    print(f\"   PROCESSED:\")\n",
    "    print(f\"   {processed}\")\n",
    "    print(f\"   LENGTH: {len(original)} → {len(processed)} chars\")\n",
    "    print(\"-\" * 80)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Preprocessing Statistics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Calculate preprocessing effects\n",
    "sample_size = 1000\n",
    "sample_df = df.sample(n=sample_size, random_state=42)\n",
    "\n",
    "# Original lengths\n",
    "original_lengths = sample_df['text'].apply(len)\n",
    "original_words = sample_df['text'].apply(lambda x: len(x.split()))\n",
    "\n",
    "# Process\n",
    "processed = sample_df['text'].apply(preprocessor.preprocess)\n",
    "processed_lengths = processed.apply(len)\n",
    "processed_words = processed.apply(lambda x: len(x.split()))\n",
    "\n",
    "# Statistics\n",
    "print(\"PREPROCESSING STATISTICS\")\n",
    "print(\"=\"*80)\n",
    "print(f\"\\nCharacter Count:\")\n",
    "print(f\"  Original:  Mean={original_lengths.mean():.1f}, Std={original_lengths.std():.1f}\")\n",
    "print(f\"  Processed: Mean={processed_lengths.mean():.1f}, Std={processed_lengths.std():.1f}\")\n",
    "print(f\"  Reduction: {(1 - processed_lengths.mean()/original_lengths.mean())*100:.1f}%\")\n",
    "\n",
    "print(f\"\\nWord Count:\")\n",
    "print(f\"  Original:  Mean={original_words.mean():.1f}, Std={original_words.std():.1f}\")\n",
    "print(f\"  Processed: Mean={processed_words.mean():.1f}, Std={processed_words.std():.1f}\")\n",
    "print(f\"  Reduction: {(1 - processed_words.mean()/original_words.mean())*100:.1f}%\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize preprocessing effects\n",
    "fig, axes = plt.subplots(2, 2, figsize=(15, 10))\n",
    "\n",
    "# Character count comparison\n",
    "axes[0, 0].hist(original_lengths, bins=30, alpha=0.7, label='Original', color='red')\n",
    "axes[0, 0].hist(processed_lengths, bins=30, alpha=0.7, label='Processed', color='green')\n",
    "axes[0, 0].set_title('Character Count Distribution', fontsize=14, fontweight='bold')\n",
    "axes[0, 0].set_xlabel('Character Count')\n",
    "axes[0, 0].set_ylabel('Frequency')\n",
    "axes[0, 0].legend()\n",
    "axes[0, 0].grid(alpha=0.3)\n",
    "\n",
    "# Word count comparison\n",
    "axes[0, 1].hist(original_words, bins=30, alpha=0.7, label='Original', color='red')\n",
    "axes[0, 1].hist(processed_words, bins=30, alpha=0.7, label='Processed', color='green')\n",
    "axes[0, 1].set_title('Word Count Distribution', fontsize=14, fontweight='bold')\n",
    "axes[0, 1].set_xlabel('Word Count')\n",
    "axes[0, 1].set_ylabel('Frequency')\n",
    "axes[0, 1].legend()\n",
    "axes[0, 1].grid(alpha=0.3)\n",
    "\n",
    "# Length reduction scatter\n",
    "axes[1, 0].scatter(original_lengths, processed_lengths, alpha=0.3, color='purple')\n",
    "axes[1, 0].plot([0, max(original_lengths)], [0, max(original_lengths)], 'r--', label='No change')\n",
    "axes[1, 0].set_title('Character Count: Original vs Processed', fontsize=14, fontweight='bold')\n",
    "axes[1, 0].set_xlabel('Original Length')\n",
    "axes[1, 0].set_ylabel('Processed Length')\n",
    "axes[1, 0].legend()\n",
    "axes[1, 0].grid(alpha=0.3)\n",
    "\n",
    "# Reduction percentage\n",
    "reduction = (1 - processed_lengths / original_lengths) * 100\n",
    "axes[1, 1].hist(reduction, bins=30, color='orange', edgecolor='black')\n",
    "axes[1, 1].set_title('Text Reduction Percentage', fontsize=14, fontweight='bold')\n",
    "axes[1, 1].set_xlabel('Reduction (%)')\n",
    "axes[1, 1].set_ylabel('Frequency')\n",
    "axes[1, 1].axvline(reduction.mean(), color='red', linestyle='--', label=f'Mean: {reduction.mean():.1f}%')\n",
    "axes[1, 1].legend()\n",
    "axes[1, 1].grid(alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/preprocessing_effects.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"\\n✅ Visualization saved to: ../data/preprocessing_effects.png\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Preprocess Full Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Preprocess all texts\n",
    "print(\"🔧 Preprocessing full dataset...\\n\")\n",
    "\n",
    "df['processed_text'] = preprocessor.preprocess_corpus(\n",
    "    df['text'].values,\n",
    "    show_progress=True\n",
    ")\n",
    "\n",
    "print(\"\\n✅ Preprocessing complete!\")\n",
    "print(f\"\\nProcessed {len(df)} texts\")\n",
    "print(f\"\\nSample processed text:\")\n",
    "print(df[['text', 'processed_text']].head(3))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Feature Engineering"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Engineer additional features\n",
    "print(\"⚙️ Engineering features...\\n\")\n",
    "\n",
    "df = preprocessor.engineer_features(df, text_column='text')\n",
    "\n",
    "print(\"✅ Features engineered!\")\n",
    "print(f\"\\nNew columns: {[col for col in df.columns if col not in ['text', 'label', 'processed_text']]}\")\n",
    "print(f\"\\nFeature statistics:\")\n",
    "df[['char_count', 'word_count', 'avg_word_length', 'punctuation_count', 'uppercase_count']].describe()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Train/Validation/Test Split"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Split data\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.preprocessing import LabelEncoder\n",
    "\n",
    "print(\"✂️ Splitting data into train/val/test...\\n\")\n",
    "\n",
    "# Encode labels\n",
    "label_encoder = LabelEncoder()\n",
    "df['label_encoded'] = label_encoder.fit_transform(df['label'])\n",
    "\n",
    "# First split: train+val vs test\n",
    "train_val_df, test_df = train_test_split(\n",
    "    df,\n",
    "    test_size=0.15,\n",
    "    random_state=42,\n",
    "    stratify=df['label_encoded']\n",
    ")\n",
    "\n",
    "# Second split: train vs val\n",
    "train_df, val_df = train_test_split(\n",
    "    train_val_df,\n",
    "    test_size=0.15 / 0.85,  # 15% of total\n",
    "    random_state=42,\n",
    "    stratify=train_val_df['label_encoded']\n",
    ")\n",
    "\n",
    "print(f\"✅ Data split complete!\")\n",
    "print(f\"   Training:   {len(train_df):,} samples ({len(train_df)/len(df)*100:.1f}%)\")\n",
    "print(f\"   Validation: {len(val_df):,} samples ({len(val_df)/len(df)*100:.1f}%)\")\n",
    "print(f\"   Test:       {len(test_df):,} samples ({len(test_df)/len(df)*100:.1f}%)\")\n",
    "print(f\"   Total:      {len(df):,} samples\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize split distribution\n",
    "fig, axes = plt.subplots(1, 3, figsize=(18, 5))\n",
    "\n",
    "# Train distribution\n",
    "train_dist = train_df['label_encoded'].value_counts().sort_index()\n",
    "axes[0].bar(range(len(train_dist)), train_dist.values, color='steelblue', alpha=0.7)\n",
    "axes[0].set_title(f'Training Set Distribution\\n({len(train_df):,} samples)', fontsize=12, fontweight='bold')\n",
    "axes[0].set_xlabel('Intent Label')\n",
    "axes[0].set_ylabel('Count')\n",
    "axes[0].grid(alpha=0.3, axis='y')\n",
    "\n",
    "# Val distribution\n",
    "val_dist = val_df['label_encoded'].value_counts().sort_index()\n",
    "axes[1].bar(range(len(val_dist)), val_dist.values, color='orange', alpha=0.7)\n",
    "axes[1].set_title(f'Validation Set Distribution\\n({len(val_df):,} samples)', fontsize=12, fontweight='bold')\n",
    "axes[1].set_xlabel('Intent Label')\n",
    "axes[1].set_ylabel('Count')\n",
    "axes[1].grid(alpha=0.3, axis='y')\n",
    "\n",
    "# Test distribution\n",
    "test_dist = test_df['label_encoded'].value_counts().sort_index()\n",
    "axes[2].bar(range(len(test_dist)), test_dist.values, color='green', alpha=0.7)\n",
    "axes[2].set_title(f'Test Set Distribution\\n({len(test_df):,} samples)', fontsize=12, fontweight='bold')\n",
    "axes[2].set_xlabel('Intent Label')\n",
    "axes[2].set_ylabel('Count')\n",
    "axes[2].grid(alpha=0.3, axis='y')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../data/data_split_distribution.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"\\n✅ Split distribution visualization saved\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Save Processed Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save processed datasets\n",
    "print(\"💾 Saving processed data...\\n\")\n",
    "\n",
    "train_df.to_csv('../data/train.csv', index=False)\n",
    "val_df.to_csv('../data/val.csv', index=False)\n",
    "test_df.to_csv('../data/test.csv', index=False)\n",
    "\n",
    "print(\"✅ Data saved successfully!\")\n",
    "print(f\"   - ../data/train.csv\")\n",
    "print(f\"   - ../data/val.csv\")\n",
    "print(f\"   - ../data/test.csv\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. TF-IDF Vectorization Preview"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize vectorizer\n",
    "vectorizer = TextVectorizer(method='tfidf', max_features=5000, ngram_range=(1, 2))\n",
    "\n",
    "# Fit on training data\n",
    "X_train = vectorizer.fit_transform(train_df['processed_text'])\n",
    "X_val = vectorizer.transform(val_df['processed_text'])\n",
    "X_test = vectorizer.transform(test_df['processed_text'])\n",
    "\n",
    "print(\"✅ TF-IDF Vectorization complete!\")\n",
    "print(f\"\\nTraining matrix shape: {X_train.shape}\")\n",
    "print(f\"Validation matrix shape: {X_val.shape}\")\n",
    "print(f\"Test matrix shape: {X_test.shape}\")\n",
    "print(f\"\\nNumber of features: {len(vectorizer.get_feature_names())}\")\n",
    "print(f\"\\nTop 20 features:\")\n",
    "print(vectorizer.get_feature_names()[:20])"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Summary"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\"*80)\n",
    "print(\"PREPROCESSING PIPELINE SUMMARY\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "summary = f\"\"\"\n",
    "✅ Preprocessing Steps Completed:\n",
    "   1. Text cleaning (lowercase, remove punctuation, URLs, emails)\n",
    "   2. Tokenization using NLTK\n",
    "   3. Stopword removal\n",
    "   4. Lemmatization\n",
    "   5. Feature engineering (5 additional features)\n",
    "   6. Train/Val/Test split (70%/15%/15%)\n",
    "   7. Label encoding\n",
    "   8. TF-IDF vectorization\n",
    "\n",
    "📊 Dataset Statistics:\n",
    "   - Total samples: {len(df):,}\n",
    "   - Training: {len(train_df):,}\n",
    "   - Validation: {len(val_df):,}\n",
    "   - Test: {len(test_df):,}\n",
    "   - Number of intents: {df['label'].nunique()}\n",
    "   - TF-IDF features: {X_train.shape[1]:,}\n",
    "\n",
    "📁 Saved Files:\n",
    "   - train.csv\n",
    "   - val.csv\n",
    "   - test.csv\n",
    "   - preprocessing_effects.png\n",
    "   - data_split_distribution.png\n",
    "\n",
    "🎯 Ready for Model Training!\n",
    "\"\"\"\n",
    "\n",
    "print(summary)\n",
    "print(\"=\"*80)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}